In [1]:
import sys, os
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)
print('Project root:', project_root)

Project root: /Users/somita/Hospital_Supply_Chain_Bot


In [2]:
from src.config import DOCS_PATH, FAISS_PATH, CHUNK_SIZE, CHUNK_OVERLAP
from src.retriever import load_embeddings, get_index_info
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
print('Docs :', DOCS_PATH)
print('FAISS:', FAISS_PATH)

Docs : /Users/somita/Hospital_Supply_Chain_Bot/docs
FAISS: /Users/somita/Hospital_Supply_Chain_Bot/faiss_index


In [3]:
documents, failed = [], []
pdf_files = [f for f in DOCS_PATH.iterdir() if f.suffix == '.pdf']
print(f'Found {len(pdf_files)} PDFs')
for pdf in sorted(pdf_files):
    try:
        docs = PyPDFLoader(str(pdf)).load()
        documents.extend(docs)
        print(f'  OK   {pdf.name} ({len(docs)} pages)')
    except Exception as e:
        failed.append(pdf.name)
        print(f'  SKIP {pdf.name} — {str(e)[:60]}')
print(f'Total pages: {len(documents)} | Skipped: {len(failed)}')

Found 5 PDFs
  OK   hospital_supply_chain_data_dictionary.pdf (13 pages)
  OK   main (1).pdf (14 pages)
  OK   main.pdf (6 pages)
  OK   mds3-ch23-inventorymgmt-mar2012.pdf (24 pages)
  OK   mds3-ch44-medicalstores-mar2012.pdf (24 pages)
Total pages: 81 | Skipped: 0


In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP,
    separators=['\n\n','\n','.',' '])
chunks = splitter.split_documents(documents)
print(f'Chunks: {len(chunks)} | Avg size: {sum(len(c.page_content) for c in chunks)//len(chunks)} chars')

Chunks: 821 | Avg size: 446 chars


In [7]:
print('Loading embedding model...')
embeddings = load_embeddings()
print('Building FAISS index...')
vs = FAISS.from_documents(chunks, embeddings)
vs.save_local(str(FAISS_PATH))
print(f'Saved to: {FAISS_PATH}')

2026-04-12 17:59:45 | INFO | inventra.retriever | Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
2026-04-12 17:59:45 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-04-12 17:59:45 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
2026-04-12 17:59:45 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"


Loading embedding model...


2026-04-12 17:59:45 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-04-12 17:59:45 | INFO | sentence_transformers.base.model | Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
2026-04-12 17:59:45 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-04-12 17:59:45 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-04-12 17:59:45 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-04-12 

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-04-12 17:59:46 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-04-12 17:59:46 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-04-12 17:59:46 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-04-12 17:59:46 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/prepro

Building FAISS index...
Saved to: /Users/somita/Hospital_Supply_Chain_Bot/faiss_index


In [8]:
from src.retriever import load_retriever, search
print('Index info:', get_index_info())
_, _, retriever = load_retriever()
docs = search(retriever, 'What should I do when stock falls below reorder point?')
print(f'Test query returned {len(docs)} chunks')
print(docs[0].page_content[:200])

2026-04-12 17:59:56 | INFO | inventra.retriever | Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
2026-04-12 17:59:56 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-04-12 17:59:56 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
2026-04-12 17:59:56 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"


Index info: {'exists': True, 'files': ['index.faiss', 'index.pkl', '.ipynb_checkpoints'], 'sizes_kb': {'index.faiss': 1231.5, 'index.pkl': 550.0, '.ipynb_checkpoints': 0.1}}


2026-04-12 17:59:56 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-04-12 17:59:56 | INFO | sentence_transformers.base.model | Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
2026-04-12 17:59:56 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-04-12 17:59:56 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-04-12 17:59:57 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-04-12 

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-04-12 17:59:57 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-04-12 17:59:57 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-04-12 17:59:57 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-04-12 17:59:57 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/prepro

Test query returned 3 chunks
Section 23.5.
Reorder level. The reorder level is the quantity of remain-
ing stock that should trigger a reorder of the item. In the 
minimum-maximum ordering system, this level is called 
the minimu
